In [81]:
"""
Titanic Data Analysis and JSON Export
Author: Rita Sousa
Description: Analyze Titanic passenger data, engineer features, and export to JSON
"""
 
import pandas as pd
import numpy as np
import json
from pathlib import Path
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

Project setup complete!
Data directory: data
CSV file location: data/titanic.csv


In [82]:
import pandas as pd

df = pd.read_csv("titanic.csv")

In [83]:
print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 175

In [84]:
numeric_columns = df.select_dtypes(include='number').columns

print(numeric_columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'],
      dtype='str')


In [85]:
# Identify numeric columns
numeric_columns = df.select_dtypes(include='number')

# Calculate mean
print("Mean:")
print(numeric_columns.mean())

# Calculate median
print("\nMedian:")
print(numeric_columns.median())

# Calculate standard deviation
print("\nStandard Deviation:")
print(numeric_columns.std())

Mean:
PassengerId    446.000000
Survived         0.383838
Pclass           2.308642
Age             29.699118
SibSp            0.523008
Parch            0.381594
Fare            32.204208
dtype: float64

Median:
PassengerId    446.0000
Survived         0.0000
Pclass           3.0000
Age             28.0000
SibSp            0.0000
Parch            0.0000
Fare            14.4542
dtype: float64

Standard Deviation:
PassengerId    257.353842
Survived         0.486592
Pclass           0.836071
Age             14.526497
SibSp            1.102743
Parch            0.806057
Fare            49.693429
dtype: float64


In [86]:
# Count missing values
print("\n" + "="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)

missing_data = {}

for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percent = (missing_count / len(df)) * 100

    missing_data[col] = {
        'Missing Count': missing_count,
        'Missing Percentage': missing_percent
    }

print(pd.DataFrame(missing_data).T)


MISSING VALUES ANALYSIS
             Missing Count  Missing Percentage
PassengerId            0.0            0.000000
Survived               0.0            0.000000
Pclass                 0.0            0.000000
Name                   0.0            0.000000
Sex                    0.0            0.000000
Age                  177.0           19.865320
SibSp                  0.0            0.000000
Parch                  0.0            0.000000
Ticket                 0.0            0.000000
Fare                   0.0            0.000000
Cabin                687.0           77.104377
Embarked               2.0            0.224467


In [87]:
# Calculate missing values for each column
missing_data = df.isnull().sum()

# Sort columns by missing values (highest first)
missing_data = missing_data.sort_values(ascending=False)

# Display results
print("Columns with the most missing values:")
print(missing_data)

Columns with the most missing values:
Cabin          687
Age            177
Embarked         2
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
SibSp            0
Parch            0
Ticket           0
Fare             0
dtype: int64


In [88]:

# Create a copy of the dataframe for feature engineering
df_features = df.copy()

# Feature 1: Family Size
df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1

print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))

# Feature 2: Is Alone
df_features['IsAlone'] = (df_features['FamilySize'] == 1).astype(int)

print(df_features[['FamilySize', 'IsAlone']].head(10))

# Feature 3: Age Groups
def categorize_age(age):
    """Categorize age into groups"""
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Child'
    elif age < 30:
        return 'Young Adult'
    elif age < 50:
        return 'Adult'
    else:
        return 'Senior'

df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)

print(df_features[['Age', 'AgeGroup']].head(10))

# Analyze feature differences between survivors and non-survivors
print("\n" + "="*50)
print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")
print("="*50)

print("\nFamily Size by Survival:")
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)

# Statistical test: Do these features help differentiate?
print("\n" + "="*50)
print("FEATURE DIFFERENTIATION ANALYSIS")
print("="*50)

survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]

print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")


   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2
   FamilySize  IsAlone
0           2        0
1           2        0
2           1        1
3           2        0
4           1        1
5           1        1
6           1        1
7           5        0
8           3        0
9           2        0
    Age     AgeGroup
0  22.0  Young Adult
1  38.0        Adult
2  26.0  Young Adult
3  35.0        Adult
4  35.0        Adult
5   NaN      Unknown
6  54.0       Senior
7   2.0        Child
8  27.0  Young Adult
9  14.0        Child

FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED

Family Size by Survival:
              mean  median       std
Survived                            
0         1.883424     1.0  1.830669
1         1.938596     2.0 


## Family Size Analysis
#### #### Analyze how these features differ between survivors and non-survivors

The average family size of passengers who survived was 1.94, compared to 1.88 for passengers who did not survive. The difference was only 0.06, indicating that the average family sizes were very similar between the two groups.

This suggests that average family size alone may not be a strong indicator of passenger survival. Passengers travelling alone and passengers travelling in large families might have had very different survival rates.

In [89]:
print(df_features[['Name', 'Title']]) # Extract the passenger's title from their name

KeyError: "['Title'] not in index"

In [ ]:
# Create FareGroup feature
df_features['FareGroup'] = pd.cut(
    df_features['Fare'],
    bins=[-1, 10, 30, 100, float('inf')],
    labels=['Low', 'Medium', 'High', 'Very High']
)

# Display all passengers and their fare groups
print(df_features[['Fare', 'FareGroup']].to_string(index=False))

    Fare FareGroup
  7.2500       Low
 71.2833      High
  7.9250       Low
 53.1000      High
  8.0500       Low
  8.4583       Low
 51.8625      High
 21.0750    Medium
 11.1333    Medium
 30.0708      High
 16.7000    Medium
 26.5500    Medium
  8.0500       Low
 31.2750      High
  7.8542       Low
 16.0000    Medium
 29.1250    Medium
 13.0000    Medium
 18.0000    Medium
  7.2250       Low
 26.0000    Medium
 13.0000    Medium
  8.0292       Low
 35.5000      High
 21.0750    Medium
 31.3875      High
  7.2250       Low
263.0000 Very High
  7.8792       Low
  7.8958       Low
 27.7208    Medium
146.5208 Very High
  7.7500       Low
 10.5000    Medium
 82.1708      High
 52.0000      High
  7.2292       Low
  8.0500       Low
 18.0000    Medium
 11.2417    Medium
  9.4750       Low
 21.0000    Medium
  7.8958       Low
 41.5792      High
  7.8792       Low
  8.0500       Low
 15.5000    Medium
  7.7500       Low
 21.6792    Medium
 17.8000    Medium
 39.6875      High
  7.8000    

In [ ]:
print(df_features['FareGroup'].value_counts()) # Count passengers in each category

FareGroup
Low          336
Medium       321
High         181
Very High     53
Name: count, dtype: int64


In [ ]:
fare_survival = df_features.groupby('FareGroup', observed=True)['Survived'].mean() # Calculate survival rates for each category

print(fare_survival)

FareGroup
Low          0.199405
Medium       0.433022
High         0.535912
Very High    0.735849
Name: Survived, dtype: float64


In [ ]:
import pandas as pd
import json
from datetime import datetime

In [ ]:

# Step 3: Create Classes for JSON Export

class Passenger:
    """
    Represents a passenger with all their information.
    """

    def __init__(self, passenger_id, name, age, sex, survived, pclass,
                 fare, embarked=None, family_size=None, is_alone=None, title=None):

        # Initialize passenger attributes
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name = str(name) if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = str(sex) if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = str(embarked) if pd.notna(embarked) else None
        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone = int(is_alone) if pd.notna(is_alone) else None
        self.title = str(title) if pd.notna(title) else None

    def to_dict(self):
        """Convert passenger to dictionary for JSON serialization."""

        return {
            'passenger_id': self.passenger_id,
            'name': self.name,
            'age': self.age,
            'sex': self.sex,
            'survived': self.survived,
            'pclass': self.pclass,
            'fare': self.fare,
            'embarked': self.embarked,
            'family_size': self.family_size,
            'is_alone': self.is_alone,
            'title': self.title
        }


class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods for JSON export.
    """

    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []
        self._create_passengers()

    def _create_passengers(self):
        """Create Passenger objects from dataframe."""

        for idx, row in self.dataframe.iterrows():

            passenger = Passenger(
                passenger_id=row.get('PassengerId', idx),
                name=row.get('Name'),
                age=row.get('Age'),
                sex=row.get('Sex'),
                survived=row.get('Survived'),
                pclass=row.get('Pclass'),
                fare=row.get('Fare'),
                embarked=row.get('Embarked'),
                family_size=row.get('FamilySize'),
                is_alone=row.get('IsAlone'),
                title=row.get('Title')
            )

            self.passengers.append(passenger)

    def to_json(self, filename='titanic_data.json'):
        """Export dataset to JSON file."""

        data = {
            'metadata': {
                'dataset_name': 'Titanic Passenger Dataset',
                'export_date': datetime.now().isoformat(),
                'total_passengers': len(self.passengers),
                'survival_rate': float(
                    self.dataframe['Survived'].mean()
                )
            },
            'passengers': [
                p.to_dict() for p in self.passengers
            ]
        }

        # Write JSON file
        with open(filename, 'w', encoding='utf-8') as file:
            json.dump(data, file, indent=2, ensure_ascii=False)

        print(f"Data exported successfully to {filename}")

        return data

    def get_summary_stats(self):
        """Get summary statistics."""

        survived_count = sum(
            1 for p in self.passengers if p.survived == 1
        )

        not_survived_count = sum(
            1 for p in self.passengers if p.survived == 0
        )

        ages = [p.age for p in self.passengers if p.age is not None]
        fares = [p.fare for p in self.passengers if p.fare is not None]

        return {
            'total_passengers': len(self.passengers),
            'survived': survived_count,
            'did_not_survive': not_survived_count,
            'average_age': round(sum(ages) / len(ages), 2) if ages else None,
            'average_fare': round(sum(fares) / len(fares), 2) if fares else None
        }


# Create dataset object and export

# Use df_features because it contains the engineered features
dataset = TitanicDataset(df_features)

# Display basic information
print("TITANIC DATASET")
print("=" * 40)
print("Total passengers:", len(dataset.passengers))

# Display summary statistics
print("\nSUMMARY STATISTICS")
print("=" * 40)

stats = dataset.get_summary_stats()

for key, value in stats.items():
    print(f"{key}: {value}")

# Export to JSON
print("\nJSON EXPORT")
print("=" * 40)

dataset.to_json('titanic_data.json')


TITANIC DATASET
Total passengers: 891

SUMMARY STATISTICS
total_passengers: 891
survived: 342
did_not_survive: 549
average_age: 29.7
average_fare: 32.2

JSON EXPORT
Data exported successfully to titanic_data.json


{'metadata': {'dataset_name': 'Titanic Passenger Dataset',
  'export_date': '2026-10-31T19:56:38.230885',
  'total_passengers': 891,
  'survival_rate': 0.3838383838383838},
 'passengers': [{'passenger_id': 1,
   'name': 'Braund, Mr. Owen Harris',
   'age': 22.0,
   'sex': 'male',
   'survived': 0,
   'pclass': 3,
   'fare': 7.25,
   'embarked': 'S',
   'family_size': 2,
   'is_alone': 0,
   'title': 'Mr'},
  {'passenger_id': 2,
   'name': 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)',
   'age': 38.0,
   'sex': 'female',
   'survived': 1,
   'pclass': 1,
   'fare': 71.2833,
   'embarked': 'C',
   'family_size': 2,
   'is_alone': 0,
   'title': 'Mrs'},
  {'passenger_id': 3,
   'name': 'Heikkinen, Miss. Laina',
   'age': 26.0,
   'sex': 'female',
   'survived': 1,
   'pclass': 3,
   'fare': 7.925,
   'embarked': 'S',
   'family_size': 1,
   'is_alone': 1,
   'title': 'Miss'},
  {'passenger_id': 4,
   'name': 'Futrelle, Mrs. Jacques Heath (Lily May Peel)',
   'age': 35.0,
   'sex':

In [ ]:
# Read the exported JSON file
with open('titanic_data.json', 'r', encoding='utf-8') as file:
    json_data = json.load(file)

# Display the first passenger
print(json_data['passengers'][0])

{'passenger_id': 1, 'name': 'Braund, Mr. Owen Harris', 'age': 22.0, 'sex': 'male', 'survived': 0, 'pclass': 3, 'fare': 7.25, 'embarked': 'S', 'family_size': 2, 'is_alone': 0, 'title': 'Mr'}


In [ ]:
print(json_data['metadata'])

{'dataset_name': 'Titanic Passenger Dataset', 'export_date': '2026-10-31T19:56:38.230885', 'total_passengers': 891, 'survival_rate': 0.3838383838383838}


In [ ]:

# ============================================================
# STEP 3: CREATE CLASSES FOR JSON EXPORT
# ============================================================

import pandas as pd
import json
from datetime import datetime


class Passenger:
    """
    Represents a passenger with all their information.
    """

    def __init__(self, passenger_id, name, age, sex, survived,
                 pclass, fare, embarked=None, family_size=None,
                 is_alone=None, title=None, age_group=None,
                 fare_group=None, has_cabin=None):

        # Initialize passenger attributes
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name = str(name) if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = str(sex) if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = str(embarked) if pd.notna(embarked) else None

        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone = int(is_alone) if pd.notna(is_alone) else None
        self.title = str(title) if pd.notna(title) else None
        self.age_group = str(age_group) if pd.notna(age_group) else None
        self.fare_group = str(fare_group) if pd.notna(fare_group) else None
        self.has_cabin = int(has_cabin) if pd.notna(has_cabin) else None

    def to_dict(self):
        """Convert passenger to dictionary for JSON serialization."""

        return {
            'passenger_id': self.passenger_id,
            'name': self.name,
            'age': self.age,
            'sex': self.sex,
            'survived': self.survived,
            'pclass': self.pclass,
            'fare': self.fare,
            'embarked': self.embarked,
            'family_size': self.family_size,
            'is_alone': self.is_alone,
            'title': self.title,
            'age_group': self.age_group,
            'fare_group': self.fare_group,
            'has_cabin': self.has_cabin
        }

class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods
    for data analysis and JSON export.
    """

    def __init__(self, dataframe):

        self.dataframe = dataframe.copy()
        self.passengers = []

        self._create_passengers()

    def _create_passengers(self):
        """Create Passenger objects from dataframe."""

        for idx, row in self.dataframe.iterrows():

            passenger = Passenger(
                passenger_id=row.get('PassengerId', idx),
                name=row.get('Name'),
                age=row.get('Age'),
                sex=row.get('Sex'),
                survived=row.get('Survived'),
                pclass=row.get('Pclass'),
                fare=row.get('Fare'),
                embarked=row.get('Embarked'),
                family_size=row.get('FamilySize'),
                is_alone=row.get('IsAlone'),
                title=row.get('Title'),
                age_group=row.get('AgeGroup'),
                fare_group=row.get('FareGroup'),
                has_cabin=row.get('HasCabin')
            )

            self.passengers.append(passenger)



    def get_summary_stats(self):
        """Calculate summary statistics."""

        survived_count = sum(
            1 for p in self.passengers if p.survived == 1
        )

        not_survived_count = sum(
            1 for p in self.passengers if p.survived == 0
        )

        ages = [p.age for p in self.passengers if p.age is not None]
        fares = [p.fare for p in self.passengers if p.fare is not None]

        return {
            'total_passengers': len(self.passengers),
            'survived': survived_count,
            'did_not_survive': not_survived_count,
            'survival_rate': round(
                survived_count / len(self.passengers), 4
            ) if self.passengers else None,
            'average_age': round(sum(ages) / len(ages), 2) if ages else None,
            'average_fare': round(sum(fares) / len(fares), 2) if fares else None
        }

    # --------------------------------------------------------
    # NUMERIC STATISTICS
    # --------------------------------------------------------

    def get_numeric_statistics(self):
        """Calculate mean, median and standard deviation."""

        numeric_data = self.dataframe.select_dtypes(include='number')

        return numeric_data.agg(
            ['mean', 'median', 'std']
        ).to_dict()

    # --------------------------------------------------------
    # MISSING VALUES ANALYSIS
    # --------------------------------------------------------

    def get_missing_values(self):
        """Calculate missing values and percentages."""

        missing_count = self.dataframe.isnull().sum()

        missing_percentage = (
            self.dataframe.isnull().sum() /
            len(self.dataframe) * 100
        ) if len(self.dataframe) > 0 else self.dataframe.isnull().sum()

        return {
            'missing_count': missing_count.to_dict(),
            'missing_percentage': missing_percentage.round(2).to_dict()
        }

    # --------------------------------------------------------
    # ENGINEERED FEATURES
    # --------------------------------------------------------

    def get_engineered_features(self):
        """Summarize engineered features."""

        features = [
            'FamilySize',
            'IsAlone',
            'AgeGroup',
            'Title',
            'FareGroup',
            'HasCabin'
        ]

        result = {}

        for col in features:
            if col in self.dataframe.columns:
                result[col] = self.dataframe[col].value_counts(
                    dropna=False
                ).to_dict()

        return result

    # --------------------------------------------------------
    # JSON EXPORT
    # --------------------------------------------------------

    def to_json(self, filename='titanic_data.json'):
        """Export all dataset information and analysis to JSON."""

        data = {
            'metadata': {
                'dataset_name': 'Titanic Passenger Dataset',
                'export_date': datetime.now().isoformat(),
                'total_passengers': len(self.passengers),
                'survival_rate': float(
                    self.dataframe['Survived'].mean()
                ) if len(self.dataframe) > 0 else None
            },

            'passengers': [
                p.to_dict() for p in self.passengers
            ],

            'summary_statistics': self.get_summary_stats(),

            'numeric_statistics': self.get_numeric_statistics(),

            'missing_values_analysis': self.get_missing_values(),

            'engineered_features': self.get_engineered_features()
        }

        # Write data to JSON file
        with open(filename, 'w', encoding='utf-8') as file:
            json.dump(
                data,
                file,
                indent=2,
                ensure_ascii=False,
                default=str,
                allow_nan=False
            )

        print(f"\nData successfully exported to {filename}")

        return data


# ============================================================
# CREATE DATASET OBJECT AND EXPORT
# ============================================================

# Use the engineered DataFrame
if 'df_features' in locals() and not df_features.empty:

    # Create TitanicDataset object
    dataset = TitanicDataset(df_features)

    # Display basic information
    print("=" * 50)
    print("TITANIC DATASET")
    print("=" * 50)

    print("Total passengers:", len(dataset.passengers))

    # Display summary statistics
    print("\nSUMMARY STATISTICS")
    print("=" * 50)

    stats = dataset.get_summary_stats()

    for key, value in stats.items():
        print(f"{key}: {value}")

    # Display numeric statistics
    print("\nNUMERIC STATISTICS")
    print("=" * 50)

    print(pd.DataFrame(dataset.get_numeric_statistics()))

    # Display missing values
    print("\nMISSING VALUES ANALYSIS")
    print("=" * 50)

    print(pd.DataFrame(dataset.get_missing_values()))

    # Display engineered features
    print("\nENGINEERED FEATURES")
    print("=" * 50)

    for feature, values in dataset.get_engineered_features().items():
        print(f"\n{feature}:")
        print(values)

    # Export everything to JSON
    dataset.to_json('titanic_data.json')




TITANIC DATASET
Total passengers: 891

SUMMARY STATISTICS
total_passengers: 891
survived: 342
did_not_survive: 549
survival_rate: 0.3838
average_age: 29.7
average_fare: 32.2

NUMERIC STATISTICS
        PassengerId  Survived    Pclass        Age     SibSp     Parch  \
mean     446.000000  0.383838  2.308642  29.699118  0.523008  0.381594   
median   446.000000  0.000000  3.000000  28.000000  0.000000  0.000000   
std      257.353842  0.486592  0.836071  14.526497  1.102743  0.806057   

             Fare  FamilySize   IsAlone  
mean    32.204208    1.904602  0.602694  
median  14.454200    1.000000  1.000000  
std     49.693429    1.613459  0.489615  

MISSING VALUES ANALYSIS
             missing_count  missing_percentage
PassengerId              0                0.00
Survived                 0                0.00
Pclass                   0                0.00
Name                     0                0.00
Sex                      0                0.00
Age                    177        

In [ ]:
print("Dataset loaded successfully!") #Check if dataset is loaded
print("Rows and columns:", df_features.shape)

print("\nFirst 5 rows:")
display(df_features.head())

Dataset loaded successfully!
Rows and columns: (891, 17)

First 5 rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,IsAlone,AgeGroup,Title,FareGroup
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2,0,Young Adult,Mr,Low
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2,0,Adult,Mrs,High
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1,1,Young Adult,Miss,Low
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,2,0,Adult,Mrs,High
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1,1,Adult,Mr,Low


In [ ]:
dataset = TitanicDataset(df_features) # Check that the class loads data correctly

print("Total passengers:", len(dataset.passengers))

# Display the first passenger
print("\nFirst passenger:")
print(dataset.passengers[0].to_dict())

Total passengers: 891

First passenger:
{'passenger_id': 1, 'name': 'Braund, Mr. Owen Harris', 'age': 22.0, 'sex': 'male', 'survived': 0, 'pclass': 3, 'fare': 7.25, 'embarked': 'S', 'family_size': 2, 'is_alone': 0, 'title': 'Mr', 'age_group': 'Young Adult', 'fare_group': 'Low', 'has_cabin': None}


In [ ]:
stats = dataset.get_summary_stats() # Check that statistics are calculated

print("SUMMARY STATISTICS")

for key, value in stats.items():
    print(f"{key}: {value}")

SUMMARY STATISTICS
total_passengers: 891
survived: 342
did_not_survive: 549
survival_rate: 0.3838
average_age: 29.7
average_fare: 32.2


In [ ]:
missing = dataset.get_missing_values() # Check missing values

print("MISSING VALUES ANALYSIS")

print("\nMissing counts:")
print(missing['missing_count'])

print("\nMissing percentages:")
print(missing['missing_percentage'])

MISSING VALUES ANALYSIS

Missing counts:
{'PassengerId': 0, 'Survived': 0, 'Pclass': 0, 'Name': 0, 'Sex': 0, 'Age': 177, 'SibSp': 0, 'Parch': 0, 'Ticket': 0, 'Fare': 0, 'Cabin': 687, 'Embarked': 2, 'FamilySize': 0, 'IsAlone': 0, 'AgeGroup': 0, 'Title': 0, 'FareGroup': 0}

Missing percentages:
{'PassengerId': 0.0, 'Survived': 0.0, 'Pclass': 0.0, 'Name': 0.0, 'Sex': 0.0, 'Age': 19.87, 'SibSp': 0.0, 'Parch': 0.0, 'Ticket': 0.0, 'Fare': 0.0, 'Cabin': 77.1, 'Embarked': 0.22, 'FamilySize': 0.0, 'IsAlone': 0.0, 'AgeGroup': 0.0, 'Title': 0.0, 'FareGroup': 0.0}


In [ ]:
features = dataset.get_engineered_features() # Check engineered features

print("ENGINEERED FEATURES")

for feature, values in features.items():
    print(f"\n{feature}:")
    print(values)

ENGINEERED FEATURES

FamilySize:
{1: 537, 2: 161, 3: 102, 4: 29, 6: 22, 5: 15, 7: 12, 11: 7, 8: 6}

IsAlone:
{1: 537, 0: 354}

AgeGroup:
{'Young Adult': 271, 'Adult': 256, 'Unknown': 177, 'Child': 113, 'Senior': 74}

Title:
{'Mr': 517, 'Miss': 182, 'Mrs': 125, 'Master': 40, 'Dr': 7, 'Rev': 6, 'Major': 2, 'Mlle': 2, 'Col': 2, 'Don': 1, 'Mme': 1, 'Ms': 1, 'Lady': 1, 'Sir': 1, 'Capt': 1, 'the Countess': 1, 'Jonkheer': 1}

FareGroup:
{'Low': 336, 'Medium': 321, 'High': 181, 'Very High': 53}


In [ ]:
dataset.to_json('titanic_data.json') # Export the JSON file


Data successfully exported to titanic_data.json


{'metadata': {'dataset_name': 'Titanic Passenger Dataset',
  'export_date': '2026-10-31T20:21:50.731871',
  'total_passengers': 891,
  'survival_rate': 0.3838383838383838},
 'passengers': [{'passenger_id': 1,
   'name': 'Braund, Mr. Owen Harris',
   'age': 22.0,
   'sex': 'male',
   'survived': 0,
   'pclass': 3,
   'fare': 7.25,
   'embarked': 'S',
   'family_size': 2,
   'is_alone': 0,
   'title': 'Mr',
   'age_group': 'Young Adult',
   'fare_group': 'Low',
   'has_cabin': None},
  {'passenger_id': 2,
   'name': 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)',
   'age': 38.0,
   'sex': 'female',
   'survived': 1,
   'pclass': 1,
   'fare': 71.2833,
   'embarked': 'C',
   'family_size': 2,
   'is_alone': 0,
   'title': 'Mrs',
   'age_group': 'Adult',
   'fare_group': 'High',
   'has_cabin': None},
  {'passenger_id': 3,
   'name': 'Heikkinen, Miss. Laina',
   'age': 26.0,
   'sex': 'female',
   'survived': 1,
   'pclass': 3,
   'fare': 7.925,
   'embarked': 'S',
   'family_size':

In [ ]:
# Validate the JSON file
with open('titanic_data.json', 'r', encoding='utf-8') as file: 
    json_data = json.load(file)

# Check that JSON is valid
print("JSON validation passed!")

# Check JSON structure
print("\nJSON sections:")
print(json_data.keys())

# Check number of passengers
print("\nNumber of passengers:")
print(len(json_data['passengers']))

# Check metadata
print("\nMetadata:")
print(json_data['metadata'])

JSON validation passed!

JSON sections:
dict_keys(['metadata', 'passengers', 'summary_statistics', 'numeric_statistics', 'missing_values_analysis', 'engineered_features'])

Number of passengers:
891

Metadata:
{'dataset_name': 'Titanic Passenger Dataset', 'export_date': '2026-10-31T20:21:50.731871', 'total_passengers': 891, 'survival_rate': 0.3838383838383838}


In [95]:

# Additional validation: Load and inspect JSON

import json
import os

JSON_FILE = 'titanic_data.json'

with open(JSON_FILE, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

# Print summary of JSON data and verify content

print("=" * 50)
print("JSON FILE VALIDATION")
print("=" * 50)

# 1. Check JSON sections
print("\nJSON sections:")
print(json_data.keys())

# 2. Display metadata
print("\nDataset Metadata:")
print(json_data['metadata'])

# 3. Count passengers
print("\nTotal passengers:")
print(len(json_data['passengers']))

# 4. Display first passenger
print("\nFirst passenger:")
print(json.dumps(json_data['passengers'][0], indent=4))

# 5. Display summary statistics
print("\nSummary Statistics:")
print(json_data['summary_statistics'])

# 6. Display numeric statistics
print("\nNumeric Statistics:")
print(json.dumps(json_data['numeric_statistics'], indent=4))

# 7. Display missing values analysis
print("\nMissing Values Analysis:")
print(json.dumps(json_data['missing_values_analysis'], indent=4))

# 8. Check engineered features
print("\nEngineered Features:")
print(json.dumps(json_data['engineered_features'], indent=4))

# 9. Verify JSON structure
required_sections = [
    'metadata',
    'passengers',
    'summary_statistics',
    'numeric_statistics',
    'missing_values_analysis',
    'engineered_features'
]

for section in required_sections:
    assert section in json_data, f"Missing section: {section}"

print("\nAll required JSON sections are present!")

# 10. Final validation
assert len(json_data['passengers']) > 0
assert json_data['metadata']['total_passengers'] == len(json_data['passengers'])

print("\nJSON VALIDATION PASSED!")
print("File size:", os.path.getsize(JSON_FILE), "bytes")


JSON FILE VALIDATION

JSON sections:
dict_keys(['metadata', 'passengers', 'summary_statistics', 'numeric_statistics', 'missing_values_analysis', 'engineered_features'])

Dataset Metadata:
{'dataset_name': 'Titanic Passenger Dataset', 'export_date': '2026-10-31T20:21:50.731871', 'total_passengers': 891, 'survival_rate': 0.3838383838383838}

Total passengers:
891

First passenger:
{
    "passenger_id": 1,
    "name": "Braund, Mr. Owen Harris",
    "age": 22.0,
    "sex": "male",
    "survived": 0,
    "pclass": 3,
    "fare": 7.25,
    "embarked": "S",
    "family_size": 2,
    "is_alone": 0,
    "title": "Mr",
    "age_group": "Young Adult",
    "fare_group": "Low",
    "has_cabin": null
}

Summary Statistics:
{'total_passengers': 891, 'survived': 342, 'did_not_survive': 549, 'survival_rate': 0.3838, 'average_age': 29.7, 'average_fare': 32.2}

Numeric Statistics:
{
    "PassengerId": {
        "mean": 446.0,
        "median": 446.0,
        "std": 257.3538420152301
    },
    "Survived"

In [97]:

import json
import os

# Step 1: Load the JSON file
JSON_FILE = 'titanic_data.json'

with open(JSON_FILE, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

print("1. JSON file loaded successfully!")

# Step 2: Verify all expected keys
expected_keys = [
    'metadata',
    'passengers',
    'summary_statistics',
    'numeric_statistics',
    'missing_values_analysis',
    'engineered_features'
]

for key in expected_keys:
    assert key in json_data, f"Missing key: {key}"

print("\n2. All expected JSON keys are present!")

# Step 3: Verify passenger records
json_passengers = len(json_data['passengers'])
original_passengers = len(df_features)

print("\n3. PASSENGER RECORD VALIDATION")
print("JSON passenger records:", json_passengers)
print("Original dataset records:", original_passengers)

assert json_passengers == original_passengers

print("Passenger count is correct!")

# Step 4: Verify statistics
assert 'summary_statistics' in json_data
assert 'numeric_statistics' in json_data

print("\n4. STATISTICS VALIDATION")
print("Summary statistics:")
print(json_data['summary_statistics'])

print("\nNumeric statistics:")
print(json_data['numeric_statistics'])

# Step 5: Verify missing value information
missing_info = json_data['missing_values_analysis']

assert 'missing_count' in missing_info
assert 'missing_percentage' in missing_info

print("\n5. MISSING VALUES VALIDATION")
print("Missing value counts:")
print(missing_info['missing_count'])

print("\nMissing value percentages:")
print(missing_info['missing_percentage'])

# Step 6: Verify engineered features
assert 'engineered_features' in json_data

print("\n6. FEATURE VALIDATION")
print(json_data['engineered_features'].keys())

# Step 7: Final validation
print("\n" + "=" * 50)
print("FINAL VALIDATION RESULTS")
print("=" * 50)

print("PASS - JSON loaded without errors")
print("PASS - All expected keys are present")
print("PASS - Passenger count is correct")
print("PASS - Statistics are included")
print("PASS - Missing value information is included")
print("PASS - Engineered features are included")

print("\nALL CHECKPOINTS PASSED!")


1. JSON file loaded successfully!

2. All expected JSON keys are present!

3. PASSENGER RECORD VALIDATION
JSON passenger records: 891
Original dataset records: 891
Passenger count is correct!

4. STATISTICS VALIDATION
Summary statistics:
{'total_passengers': 891, 'survived': 342, 'did_not_survive': 549, 'survival_rate': 0.3838, 'average_age': 29.7, 'average_fare': 32.2}

Numeric statistics:
{'PassengerId': {'mean': 446.0, 'median': 446.0, 'std': 257.3538420152301}, 'Survived': {'mean': 0.3838383838383838, 'median': 0.0, 'std': 0.4865924542648575}, 'Pclass': {'mean': 2.308641975308642, 'median': 3.0, 'std': 0.836071240977049}, 'Age': {'mean': 29.69911764705882, 'median': 28.0, 'std': 14.526497332334042}, 'SibSp': {'mean': 0.5230078563411896, 'median': 0.0, 'std': 1.1027434322934317}, 'Parch': {'mean': 0.38159371492704824, 'median': 0.0, 'std': 0.8060572211299483}, 'Fare': {'mean': 32.204207968574636, 'median': 14.4542, 'std': 49.6934285971809}, 'FamilySize': {'mean': 1.904601571268238, 

In [100]:
dataset.to_json('data/titanic_processed.json')


Data successfully exported to data/titanic_processed.json


{'metadata': {'dataset_name': 'Titanic Passenger Dataset',
  'export_date': '2026-10-31T20:39:55.933958',
  'total_passengers': 891,
  'survival_rate': 0.3838383838383838},
 'passengers': [{'passenger_id': 1,
   'name': 'Braund, Mr. Owen Harris',
   'age': 22.0,
   'sex': 'male',
   'survived': 0,
   'pclass': 3,
   'fare': 7.25,
   'embarked': 'S',
   'family_size': 2,
   'is_alone': 0,
   'title': 'Mr',
   'age_group': 'Young Adult',
   'fare_group': 'Low',
   'has_cabin': None},
  {'passenger_id': 2,
   'name': 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)',
   'age': 38.0,
   'sex': 'female',
   'survived': 1,
   'pclass': 1,
   'fare': 71.2833,
   'embarked': 'C',
   'family_size': 2,
   'is_alone': 0,
   'title': 'Mrs',
   'age_group': 'Adult',
   'fare_group': 'High',
   'has_cabin': None},
  {'passenger_id': 3,
   'name': 'Heikkinen, Miss. Laina',
   'age': 26.0,
   'sex': 'female',
   'survived': 1,
   'pclass': 3,
   'fare': 7.925,
   'embarked': 'S',
   'family_size':

In [101]:
# ==========================================
# DATA PROCESSING RESULTS
# ==========================================

print("=" * 60)
print("1. STATISTICS CALCULATIONS")
print("=" * 60)

# Summary statistics
print("\nSummary Statistics:")
print(dataset.get_summary_stats())

# Mean, median and standard deviation
print("\nNumeric Statistics:")
numeric_stats = pd.DataFrame(dataset.get_numeric_statistics())
print(numeric_stats)

1. STATISTICS CALCULATIONS

Summary Statistics:
{'total_passengers': 891, 'survived': 342, 'did_not_survive': 549, 'survival_rate': 0.3838, 'average_age': 29.7, 'average_fare': 32.2}

Numeric Statistics:
        PassengerId  Survived    Pclass        Age     SibSp     Parch  \
mean     446.000000  0.383838  2.308642  29.699118  0.523008  0.381594   
median   446.000000  0.000000  3.000000  28.000000  0.000000  0.000000   
std      257.353842  0.486592  0.836071  14.526497  1.102743  0.806057   

             Fare  FamilySize   IsAlone  
mean    32.204208    1.904602  0.602694  
median  14.454200    1.000000  1.000000  
std     49.693429    1.613459  0.489615  


In [102]:
print("\n" + "=" * 60)
print("2. MISSING VALUE ANALYSIS")
print("=" * 60)

missing = dataset.get_missing_values()

missing_df = pd.DataFrame({
    'Missing Count': missing['missing_count'],
    'Missing Percentage': missing['missing_percentage']
})

print(missing_df)



2. MISSING VALUE ANALYSIS
             Missing Count  Missing Percentage
PassengerId              0                0.00
Survived                 0                0.00
Pclass                   0                0.00
Name                     0                0.00
Sex                      0                0.00
Age                    177               19.87
SibSp                    0                0.00
Parch                    0                0.00
Ticket                   0                0.00
Fare                     0                0.00
Cabin                  687               77.10
Embarked                 2                0.22
FamilySize               0                0.00
IsAlone                  0                0.00
AgeGroup                 0                0.00
Title                    0                0.00
FareGroup                0                0.00


In [103]:
print("\n" + "=" * 60)
print("3. FEATURE ENGINEERING RESULTS")
print("=" * 60)

features = dataset.get_engineered_features()

for feature, values in features.items():
    print(f"\n{feature}:")
    print(values)



3. FEATURE ENGINEERING RESULTS

FamilySize:
{1: 537, 2: 161, 3: 102, 4: 29, 6: 22, 5: 15, 7: 12, 11: 7, 8: 6}

IsAlone:
{1: 537, 0: 354}

AgeGroup:
{'Young Adult': 271, 'Adult': 256, 'Unknown': 177, 'Child': 113, 'Senior': 74}

Title:
{'Mr': 517, 'Miss': 182, 'Mrs': 125, 'Master': 40, 'Dr': 7, 'Rev': 6, 'Major': 2, 'Mlle': 2, 'Col': 2, 'Don': 1, 'Mme': 1, 'Ms': 1, 'Lady': 1, 'Sir': 1, 'Capt': 1, 'the Countess': 1, 'Jonkheer': 1}

FareGroup:
{'Low': 336, 'Medium': 321, 'High': 181, 'Very High': 53}


In [104]:
print("\n" + "=" * 60)
print("4. JSON EXPORT CONFIRMATION")
print("=" * 60)

import os
import json

# Export dataset
os.makedirs('data', exist_ok=True)

dataset.to_json('data/titanic_processed.json')

# Verify JSON file
JSON_FILE = 'data/titanic_processed.json'

with open(JSON_FILE, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

print("\nJSON file created successfully!")
print("File location:", JSON_FILE)
print("Total passengers exported:", len(json_data['passengers']))
print("JSON validation: PASSED")

print("\n" + "=" * 60)
print("ALL DATA PROCESSING TASKS COMPLETED!")
print("=" * 60)


4. JSON EXPORT CONFIRMATION

Data successfully exported to data/titanic_processed.json

JSON file created successfully!
File location: data/titanic_processed.json
Total passengers exported: 891
JSON validation: PASSED

ALL DATA PROCESSING TASKS COMPLETED!


In [6]:
%cd ~/Desktop/Titanic_Project/lab-data-manipulation-json

/Users/ritasousa/Desktop/Titanic_Project/lab-data-manipulation-json


In [7]:
!pwd

/Users/ritasousa/Desktop/Titanic_Project/lab-data-manipulation-json


In [8]:
!git status

On branch feature/my-solution
Your branch is up to date with 'origin/feature/my-solution'.

nothing to commit, working tree clean


In [10]:
import os
os.getcwd()

'/Users/ritasousa/Desktop/Titanic_Project/lab-data-manipulation-json'

In [11]:
!git log --oneline -5

6f4894b (HEAD -> feature/my-solution, origin/main, origin/feature/my-solution, origin/HEAD, main) Initial lab scaffold: instructions, rubric, README


In [13]:
!git diff main..feature/my-solution --stat

In [14]:
!git status --short

In [15]:
from pathlib import Path

for f in Path(".").glob("*.ipynb"):
    print(f.name)

In [16]:
!git ls-files "*.ipynb"

In [17]:
import os
print(os.getcwd())

/Users/ritasousa/Desktop/Titanic_Project/lab-data-manipulation-json


In [18]:
import os
for root, dirs, files in os.walk(os.getcwd()):
    for file in files:
        if file.endswith(".ipynb"):
            print(os.path.join(root, file))

In [19]:
import os

print("Current folder:", os.getcwd())
print("Files in this folder:", os.listdir())

Current folder: /Users/ritasousa/Desktop/Titanic_Project/lab-data-manipulation-json
Files in this folder: ['README.md', 'rubric.md', '.git', 'instructions.md']


In [21]:
from pathlib import Path

for f in Path.home().joinpath("Desktop").rglob("*.ipynb"):
    print(f)

/Users/ritasousa/Desktop/AI Consulting/homeguard_system.ipynb
/Users/ritasousa/Desktop/AI Consulting/first.notebook.ipynb
/Users/ritasousa/Desktop/Titanic_Project/titanic_analysis.ipynb


In [25]:
import shutil
from pathlib import Path

source = Path.home() / "Desktop/Titanic_Project/titanic_analysis.ipynb"

destination = (
    Path.home()
    / "Desktop/Titanic_Project/lab-data-manipulation-json/titanic_analysis.ipynb"
)

shutil.copy2(source, destination)

print("Notebook copied successfully!")

Notebook copied successfully!


In [26]:
%cd ~/Desktop/Titanic_Project/lab-data-manipulation-json
!git status

/Users/ritasousa/Desktop/Titanic_Project/lab-data-manipulation-json
On branch feature/my-solution
Your branch is up to date with 'origin/feature/my-solution'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	titanic_analysis.ipynb

nothing added to commit but untracked files present (use "git add" to track)


In [27]:
!git add titanic_analysis.ipynb

In [28]:
!git commit -m "Add completed Titanic analysis notebook"

[feature/my-solution 4992a5f] Add completed Titanic analysis notebook
 1 file changed, 38012 insertions(+)
 create mode 100644 titanic_analysis.ipynb


In [29]:
!git push origin feature/my-solution

Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 8 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 101.06 KiB | 7.77 MiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/RitSousa/lab-data-manipulation-json.git
   6f4894b..4992a5f  feature/my-solution -> feature/my-solution
